### LOAD ATT&CK DATA

In [1]:
import json
from pathlib import Path

project = Path(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx"
)

attack_file = project / "data" / "raw" / "attack" / "enterprise-attack-19.1.json"

with open(attack_file, "r", encoding="utf-8") as f:
    attack_data = json.load(f)

objects = attack_data["objects"]

print(len(objects))

25843


### EXTRACT MITIGATIONS

In [2]:
mitigations = []

for obj in objects:

    if obj.get("type") == "course-of-action":

        mitigations.append(obj)

len(mitigations)

268

### one Mitigations 

In [3]:
mitigations[0]

{'type': 'course-of-action',
 'spec_version': '2.1',
 'id': 'course-of-action--00d7d21b-69d6-4797-88a2-c86f3fc97651',
 'created': '2018-10-17T00:14:20.652Z',
 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5',
 'external_references': [{'source_name': 'mitre-attack',
   'url': 'https://attack.mitre.org/mitigations/T1174',
   'external_id': 'T1174'},
  {'source_name': 'Microsoft Install Password Filter n.d',
   'description': 'Microsoft. (n.d.). Installing and Registering a Password Filter DLL. Retrieved November 21, 2017.',
   'url': 'https://msdn.microsoft.com/library/windows/desktop/ms721766.aspx'}],
 'object_marking_refs': ['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'],
 'modified': '2025-04-18T17:59:39.912Z',
 'name': 'Password Filter DLL Mitigation',
 'description': 'Ensure only valid password filters are registered. Filter DLLs must be present in Windows installation directory (<code>C:\\Windows\\System32\\</code> by default) of a domain controller a

### BUILD MITIGATION LOOKUP

In [4]:
mitigation_lookup = {}

for m in mitigations:

    mitigation_lookup[m["id"]] = m["name"]

len(mitigation_lookup)

268

### EXTRACT RELATIONSHIPS
- Technique → Mitigation

In [5]:
relationships = []

for obj in objects:

    if obj.get("type") == "relationship":

        relationships.append(obj)

len(relationships)

21025

### FIND MITIGATION RELATION TYPES

In [6]:
relation_types = set()

for rel in relationships:

    relation_types.add(
        rel.get("relationship_type")
    )

relation_types

{'attributed-to',
 'detects',
 'mitigates',
 'revoked-by',
 'subtechnique-of',
 'uses'}

### BUILD TECHNIQUE LOOKUP

In [7]:
technique_lookup = {}

for obj in objects:

    if obj.get("type") == "attack-pattern":

        if "external_references" in obj:

            for ref in obj["external_references"]:

                if ref.get("source_name") == "mitre-attack":

                    technique_lookup[obj["id"]] = ref.get("external_id")

print("Techniques:", len(technique_lookup))

Techniques: 858


#### EXTRACT ATT&CK → MITIGATION EDGES

In [8]:
mitigation_edges = []

for rel in relationships:

    if rel.get("relationship_type") == "mitigates":

        source = rel.get("source_ref")
        target = rel.get("target_ref")

        if (
            source in mitigation_lookup
            and target in technique_lookup
        ):

            mitigation_edges.append([
                technique_lookup[target],
                "mitigated_by",
                mitigation_lookup[source]
            ])

print(len(mitigation_edges))

1448


### Results

In [9]:
mitigation_edges[:10]

[['T1659', 'mitigated_by', 'Restrict Web-Based Content'],
 ['T1595', 'mitigated_by', 'Pre-compromise'],
 ['T1205.001', 'mitigated_by', 'Filter Network Traffic'],
 ['T1566.002', 'mitigated_by', 'Software Configuration'],
 ['T1053.005', 'mitigated_by', 'Privileged Account Management'],
 ['T1580', 'mitigated_by', 'User Account Management'],
 ['T1542', 'mitigated_by', 'Limit Access to Resource Over Network'],
 ['T1599.001', 'mitigated_by', 'Password Policies'],
 ['T1505', 'mitigated_by', 'Code Signing'],
 ['T1591', 'mitigated_by', 'Pre-compromise']]

------------

### SAVE AS CSV

In [10]:
import pandas as pd

mitigation_df = pd.DataFrame(
    mitigation_edges,
    columns=["source", "relation", "target"]
)

mitigation_df.head()

,source,relation,target
0,T1659,mitigated_by,Restrict Web-Based Content
1,T1595,mitigated_by,Pre-compromise
2,T1205.001,mitigated_by,Filter Network Traffic
3,T1566.002,mitigated_by,Software Configuration
4,T1053.005,mitigated_by,Privileged Account Management


### Save

In [11]:
output_file = (
    project
    / "data"
    / "processed"
    / "attack_mitigation_edges.csv"
)

mitigation_df.to_csv(
    output_file,
    index=False
)

print("saved")

saved
